# Hiệu năng pipeline

Các số đo dưới đây thuộc những phiên resume đã nộp. Phạm vi giữa các phiên khác nhau; trạng thái cache, retry và thời gian chờ mạng cũng khác. Bảng không được dùng để xếp hạng tốc độ giữa thành viên hoặc thay benchmark fresh toàn assignment.

## Các phiên được ghi nhận

Mỗi run ID chỉ được tính một lần, kể cả khi cùng run xuất hiện trong ZIP cũ và ZIP đã sửa. Audit được chạy sau phiên benchmark không thay đổi phạm vi hoặc trạng thái kết thúc của phiên ấy.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
sys.path.insert(0, str(ROOT / 'utils'))
from preliminary_results import (load_members, progress_rows, assignment_outcomes,
    quality_rows, error_groups, performance_rows, run_range, session_members, coverage_rows, corpus_metrics, snapshot_rows, percentage, paper_timings)
members = load_members(ROOT)
sessions = session_members(ROOT, members)
def table(rows):
    frame = pd.DataFrame(rows, dtype=object)
    frame = frame.where(frame.notna(), 'Chưa đủ bằng chứng')
    def number(value):
        if isinstance(value, float):
            return f'{value:,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
        if isinstance(value, int) and not isinstance(value, bool):
            return f'{value:,}'.replace(',', '.')
        return str(value)
    display(HTML(frame.map(number).to_html(index=False, escape=True, border=0, classes='dataframe')))


In [ ]:
table([{'Phiên': s['session'], 'Phạm vi': run_range(s['benchmark']),
        'Wall time (giây)': s['benchmark']['wall_seconds'],
        'Trạng thái': 'Bị ngắt' if s['benchmark']['exit_code']==130 else ('Exit 0' if s['benchmark']['exit_code']==0 else 'Có lỗi'),
        'Số mẫu RAM': s['benchmark']['sample_count']} for s in sessions])

TV1 bổ sung phiên resume ngày 07/11 UTC trên toàn dải 5.000 ID, chạy Colab CPU-only. Phiên có mẫu RAM và timing từng paper nhưng dùng lại nhiều checkpoint. Exit 1 được giữ đúng artifact; audit sau phiên không đổi trạng thái kết thúc phép đo.

## Thời gian xử lý

Tổng wall time các phiên đã nộp bao gồm chi phí xử lý lại ID trong chính những phiên đó. Đếm trùng cùng một run ID sẽ làm sai tổng, còn việc một ID được chạy lại ở hai phiên là chi phí thực tế. Lịch sử hiện có chưa đầy đủ, nên tổng này không phải thời gian hoàn thành toàn bộ assignment. Thời gian nghỉ/ngắt máy cũng không nằm trong tổng wall time hoạt động.

Các mốc bắt đầu/kết thúc bao quanh phép đo có thể khác trường wall time; cần làm rõ biên đo và overhead trước benchmark tổng cuối cùng.

In [ ]:
table([{'Thành viên': member, 'Số phiên có tệp': len(subset),
        'Tổng wall time đã nộp (giây)': sum(s['benchmark']['wall_seconds'] for s in subset)}
       for member in sorted({s['member'] for s in sessions})
       for subset in [[s for s in sessions if s['member']==member]]])
timings = []
for s in sessions:
    events=[json.loads(line) for line in (s['directory']/'timings.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    disc=[e for e in events if e['kind']=='discovery']; paper=[e for e in events if e['kind']=='paper']
    timings.append({'Phiên': s['session'], 'Discovery (tổng giây event)': sum(e['elapsed_seconds'] for e in disc) if disc else None,
        'Paper event hoàn tất': len(paper),
        'Giây/paper event': sum(e['elapsed_seconds'] for e in paper)/len(paper) if paper else None,
        'Event dùng cache': sum(e.get('cached') is True for e in events)})
table(timings)

Paper event có thể bao gồm thời gian các stage, nên không cộng tất cả loại event thành runtime. Event đang dở khi ngắt có thể chưa được ghi. Trung bình của event trong một lượt resume chưa phải thời gian trung bình xử lý mới một paper.

## RAM và dung lượng output

RAM là tổng RSS quan sát trên cây tiến trình. Phiên không có khoảng lấy mẫu không đủ cơ sở diễn giải peak/mean đại diện. Disk đo toàn bộ output tại mốc tương ứng, có thể gồm dữ liệu tích lũy từ các phiên trước. Final disk không phải riêng kích thước corpus TeX/BibTeX.

In [ ]:
rows=[]
for s in sessions:
    b=s['benchmark'];r=b['ram'];d=b['disk'];usable=b['sample_count']>=2 and r['observed_seconds']>0
    rows.append({'Phiên':s['session'], 'Peak RSS (MiB)':r['peak_rss_bytes']/1024**2 if usable else 'Không đủ mẫu',
        'Mean RSS (MiB)':r['time_weighted_mean_rss_bytes']/1024**2 if usable and r['time_weighted_mean_rss_bytes'] is not None else 'Không đủ mẫu',
        'Peak output (GiB)':d['peak_observed_bytes']/1024**3 if d.get('peak_observed_bytes') is not None else None,
        'Final output (GiB)':d['final_bytes']/1024**3 if d.get('final_bytes') is not None else None})
table(rows)
chosen=next(s for s in sessions if s['member']=='TV3')
samples=[json.loads(line) for line in (chosen['directory']/'samples.jsonl').read_text().splitlines() if line.strip()]
if len(samples)>1:
    fig,ax=plt.subplots(figsize=(8,4));ax.plot([v['elapsed_seconds']/60 for v in samples],[v['rss_bytes']/1024**2 for v in samples])
    ax.set_xlabel('Thời gian lấy mẫu (phút)');ax.set_ylabel('Tổng RSS (MiB)');ax.set_title('RAM trong phiên resume TV3 ngày 03/11')
    plt.tight_layout();plt.show()

Biểu đồ mô tả một phiên có mẫu, chưa đại diện toàn bộ lịch sử thu thập. Không thể khôi phục runtime/RAM bị thiếu từ corpus cuối. Đỉnh ngắn có thể bị bỏ lỡ, shared pages có thể bị đếm nhiều lần khi cộng RSS, và quét file trên Drive có overhead.

## Thời gian từng paper trong các phiên đã đo

CSV được tạo khi build từ paper event, có run ID để phân biệt paper được xử lý lại qua nhiều phiên. Thời gian có thể gồm kiểm tra cache và retry; không dùng làm thời gian crawl mới của paper đó.

In [ ]:
timing_frame=pd.DataFrame(paper_timings(sessions))
output=ROOT/'_static/generated/ms1-paper-timings.csv'
output.parent.mkdir(parents=True,exist_ok=True)
timing_frame.to_csv(output,index=False)
display(HTML('<a href="../_static/generated/ms1-paper-timings.csv" download>Tải CSV thời gian từng paper theo run</a>'))
table(timing_frame.head(10).to_dict('records'))

## Mức đáp ứng bằng chứng hiệu năng

Phiên TV1 toàn dải cung cấp wall time, thời gian Discovery, thời gian từng paper và RAM cây tiến trình trên Colab CPU-only cho một lượt resume. Các phiên TV3/TV4 bổ sung số đo hoạt động thực tế. TV2/TV5 chỉ có 0–1 mẫu RAM nên không công bố peak/mean đại diện.

Khoảng lấy mẫu có thể dài hơn cấu hình vì quét output trên Drive. Disk peak là cực đại tại các lần quan sát, không bảo đảm đã thấy mọi đỉnh. Final output gồm raw, corpus, checkpoint và sidecar. Lịch sử chưa đủ để khôi phục tổng runtime/RAM/disk peak của quá trình thu thập ban đầu; không ước lượng các giá trị này từ cumulative.